In [2]:
# numpy and pandas for data manipulation
import numpy as np
import pandas as pd 
import lightgbm as lgb

# File system manangement
import os

# Suppress warnings 
import warnings
warnings.filterwarnings('ignore')

# matplotlib and seaborn for plotting
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import norm, skew
from tqdm import tqdm_notebook as tqdm
from copy import copy
from multiprocessing import Pool

from sklearnex import patch_sklearn
patch_sklearn()  # patches scikit-learn algorithms
# from sklearnex import unpatch_sklearn
# unpatch_sklearn()




X_train = pd.read_csv('../data/X_train_encoded.csv')
X_test  = pd.read_csv('../data/X_test_encoded.csv')
y_train = pd.read_csv('../data/y_train.csv')
y_test = pd.read_csv('../data/y_test.csv')

# Map target to numeric
y_train_numeric = y_train['target']
y_test_numeric = y_test['target']


Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


# This is stacking lgbm with CNN. latter change it to stacking multiple different models

Important: For your real submission or final model, you should train on the full X_train, not a subset. Subsetting is just for experimentation or debugging.

In [3]:
use_experimental = False
if use_experimental:
    np.random.seed(42)
    
    # Make sure train_length is defined before creating indices
    train_length = 150000
    total_rows = len(X_train)  # number of rows in your original X_train
    indices = np.arange(total_rows)
    np.random.shuffle(indices)
    
    # Split indices
    indices_train = indices[:train_length]
    indices_test = indices[train_length:]
    
    # Create the experimental train/test splits
    X_test = X_train.iloc[indices_test].copy()
    X_train = X_train.iloc[indices_train].copy()
    
    # Set targets
    target_train = X_train['target']
    target_test = X_test['target']
    

# Combine for certain uses
X_all = pd.concat([X_train, X_test])
print("X_all shape:", X_all.shape)


X_all shape: (200000, 208)



# Feature Engineering

Here I calculate the unique counts of each faeture seaparately. Based on that I also calculate the density by smoothing the counts and also the deviation as counts/density.

In [4]:
import numpy as np
import pandas as pd
import scipy.ndimage
from tqdm import tqdm

sigma_fac = 0.001
sigma_base = 4
eps = 1e-8  # small epsilon to avoid division by zero

def get_count_single(X_all, features):
    """
    Compute count, smoothed density, and deviation features for numeric columns in X_all.
    Returns augmented X_all and the new feature names.
    """
    features_count = np.zeros((X_all.shape[0], len(features)))
    features_density = np.zeros((X_all.shape[0], len(features)))
    features_deviation = np.zeros((X_all.shape[0], len(features)))
    
    sigmas = []

    for i, var in enumerate(tqdm(features)):
        # Convert to integer representation for counting
        X_all_var_int = (X_all[var].values * 10000).round().astype(int)
        lo = X_all_var_int.min()
        X_all_var_int -= lo
        hi = X_all_var_int.max() + 1

        # Count occurrences
        counts_all = np.bincount(X_all_var_int, minlength=hi).astype(float)

        # Smooth counts with Gaussian filter
        sigma_scaled = counts_all.shape[0] * sigma_fac
        sigma = np.power(sigma_base**2 * sigma_scaled, 1/3)
        sigmas.append(sigma)
        counts_all_smooth = scipy.ndimage.gaussian_filter1d(counts_all, sigma)

        # Deviation
        deviation = counts_all / (counts_all_smooth + eps)

        # Map counts, density, deviation back to rows
        features_count[:, i] = counts_all[X_all_var_int]
        features_density[:, i] = counts_all_smooth[X_all_var_int]
        features_deviation[:, i] = deviation[X_all_var_int]

    # Column names
    features_count_names = [f'{var}_count' for var in features]
    features_density_names = [f'{var}_density' for var in features]
    features_deviation_names = [f'{var}_deviation' for var in features]

    # Convert to DataFrames and concatenate
    X_all_count = pd.DataFrame(features_count, columns=features_count_names, index=X_all.index)
    X_all_density = pd.DataFrame(features_density, columns=features_density_names, index=X_all.index)
    X_all_deviation = pd.DataFrame(features_deviation, columns=features_deviation_names, index=X_all.index)

    X_all = pd.concat([X_all, X_all_count, X_all_density, X_all_deviation], axis=1)

    return X_all, features_count_names, features_density_names, features_deviation_names

# Suppose features are your numeric columns
features = X_all.columns.tolist()  # or select subset if needed

X_all, features_count, features_density, features_deviation = get_count_single(X_all, features)
print(X_all.shape)


100%|██████████| 208/208 [00:06<00:00, 29.85it/s]


(200000, 832)


# Standardize
I standardize all the features (or supposedly so, apparently I forgot density and deviation being in time trouble). Which is important for later NN usage.

In [5]:
from sklearn.preprocessing import StandardScaler

def get_standardized_single(X_all, features_to_scale):
    """
    Standardize numeric features in X_all.
    features_to_scale: list of lists of feature names, e.g. [features, features_count]
    """
    scaler = StandardScaler()
    
    # Flatten the list of lists into a single list
    features_flat = [var for sublist in features_to_scale for var in sublist]
    
    # Fit scaler on X_all
    scaler.fit(X_all[features_flat])
    
    # Transform
    features_scaled = scaler.transform(X_all[features_flat])
    
    # Replace in DataFrame
    X_all[features_flat] = features_scaled
    
    return X_all


# Example: scale original features and count features
features_to_scale = [features, features_count]  # features_count comes from get_count_single

X_all = get_standardized_single(X_all, features_to_scale)
print(X_all.shape)


(200000, 832)


# Setting up Dataframes
After performing FE on X_all, I split it back into train/test and delete the obsolete dataframe. The latter is a reoccuring theme in this kernel and was necessary as I often experienced memory overflow. This is also the reason why I wrote most of the code inside of functions. Shoutout to kaggle however for providing fast GPUs!

In [6]:
train_length = len(y_train)  # or len(X_train) before you created X_all

X_train = X_all.iloc[:train_length, :].copy()
X_test = X_all.iloc[train_length:, :].copy()
del X_all
import gc
gc.collect()
print(X_train.shape, X_test.shape)
X_train.index = y_train.index
X_test.index = y_test.index


(160000, 832) (40000, 832)


# LGBM
Many public kernels indicated that the features are independent, conditional on the target. For this reason I train seperate trees for each feature and their respective counts. Using a simple average (of the square root) of all tree predictors achieves around 0.9225 / 0.9205 on public/private LB.

In [7]:
features_used = [features, features_count]

params = {
    'boost_from_average':'false',
    'boost': 'gbdt',
    'feature_fraction': 1,
    'learning_rate': 0.08,
    'max_depth': -1,
    'metric':'binary_logloss',
    'num_leaves': 4,
    'num_threads': 8,
    'tree_learner': 'serial',
    'objective': 'binary',
    'reg_alpha': 2,
    'reg_lambda': 0,
    'verbosity': 1,
    'max_bin':256,
}

# reg_alpha
reg_alpha_values = [0.75, 1, 2, 3]
reg_alpha_var = [3, 0, 2, 3, 2, 0, 1, 1, 3, 2, 2, 0, 2, 0, 2, 2, 2, 1, 1, 2, 1, 2, 3, 3, 2, 1, 3, 1, 3, 2, 2, 3, 1, 1, 3, 2, 0, 1, 0, 2, 1, 1, 2, 3, 0, 3, 3, 3, 2, 0, 3, 1, 3, 1, 1, 0, 2, 2, 0, 0, 0, 1, 2, 1, 0, 1, 3, 2, 0, 2, 1, 2, 0, 0, 1, 3, 3, 1, 2, 3, 3, 2, 0, 1, 2, 3, 3, 2, 3, 3, 0, 0, 3, 0, 1, 0, 1, 0, 2, 3, 1, 0, 3, 1, 3, 2, 3, 1, 3, 3, 3, 1, 3, 2, 3, 2, 1, 0, 1, 2, 0, 3, 0, 3, 0, 3, 2, 1, 0, 0, 2, 2, 2, 0, 1, 0, 0, 2, 3, 2, 2, 1, 1, 0, 1, 2, 2, 2, 1, 0, 2, 3, 2, 3, 1, 1, 3, 1, 1, 2, 1, 2, 0, 3, 1, 3, 3, 2, 0, 1, 3, 3, 0, 1, 0, 3, 1, 3, 1, 3, 0, 3, 0, 3, 1, 0, 0, 0, 3, 0, 3, 0, 0, 2, 0, 3, 1, 0, 3, 2]

# max_bin
max_bin_values = [256, 512, 1024]
max_bin_var = [0, 0, 1, 0, 0, 0, 2, 0, 0, 2, 0, 2, 0, 0, 1, 1, 1, 0, 0, 0, 0, 2, 2, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 2, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 2, 1, 1, 1, 1, 0, 0, 0, 0, 1, 2, 1, 0, 0, 1, 2, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 2, 1, 0, 0, 0, 2, 0, 1, 1, 0, 1, 0, 0, 1, 2, 1, 2, 1, 0, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 1, 1, 0, 0, 0, 2, 0, 0, 2, 1, 0, 0, 1, 0, 1, 2, 0, 0, 0, 0, 0, 2, 2, 2, 2, 1, 1, 2, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 2, 0, 1, 0, 1, 1, 0, 2, 1, 1, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 1, 0, 1, 2, 0, 0, 0, 0, 2, 0, 0, 2, 0, 1, 1, 0, 2, 0, 0, 0, 1, 2, 0, 0, 1, 0, 2]

# learning_rate
learning_rate_values = [0.06, 0.08, 0.12]
learning_rate_var = [2, 2, 2, 1, 2, 2, 2, 0, 1, 2, 0, 2, 2, 2, 0, 2, 2, 0, 2, 1, 2, 2, 2, 2, 2, 0, 1, 0, 2, 0, 0, 2, 0, 2, 2, 2, 1, 2, 0, 0, 2, 0, 0, 1, 2, 1, 2, 0, 0, 2, 1, 2, 2, 2, 2, 0, 0, 2, 1, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 0, 2, 2, 1, 0, 1, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 1, 1, 1, 2, 0, 2, 0, 2, 0, 2, 1, 0, 0, 1, 2, 0, 2, 2, 2, 0, 2, 2, 2, 2, 1, 0, 2, 1, 2, 2, 1, 2, 0, 2, 0, 2, 2, 2, 2, 2, 2, 1, 2, 1, 0, 2, 1, 1, 2, 2, 2, 2, 0, 2, 1, 2, 1, 2, 2, 2, 2, 2, 2, 2, 1, 1, 0, 1, 2, 0, 2, 2, 0, 1, 2, 2, 2, 1, 0, 1, 2, 1, 2, 1, 1, 1, 2, 1, 2, 1, 0, 0, 2, 0, 1, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1]

# num_leaves
num_leaves_values = [3, 4, 5]
num_leaves_var = [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 2, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 2, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 2, 1, 2, 0, 0, 0, 0, 0, 1, 1, 0, 0, 2, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 2, 0, 0, 0, 0, 1, 0, 1, 2, 1, 1, 1, 0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 2, 0, 1, 0, 2, 2, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 2, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 2, 0, 0, 0, 0, 1, 0, 0, 2, 1, 0, 1, 2, 1, 1, 0, 0, 0, 2, 1, 2]

In [8]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss

n_folds = 5
early_stopping_rounds = 10
settings = [4]  # placeholder for hyperparameter tuning
np.random.seed(47)
settings_best_ind = []

def train_trees(X_train, y_train, X_test, features_used, params):
    """
    Train LightGBM trees with Stratified K-Folds, compute OOF, test, and train predictions.
    
    X_train, X_test: feature DataFrames
    y_train: target Series
    features_used: list of lists of feature names to use per target column
    params: LightGBM parameters
    """
    
    n_features = len(features_used[0])
    preds_oof = np.zeros((len(X_train), n_features))
    preds_test = np.zeros((len(X_test), n_features))
    preds_train = np.zeros((len(X_train), n_features))

    # Flatten features
    features_flat = [var for sublist in features_used for var in sublist]
    X_train_used = X_train[features_flat]
    X_test_used = X_test[features_flat]

    for i in range(n_features):
        features_train = [feature_set[i] for feature_set in features_used]
        print(f'Training on features: {features_train}')

        # Stratified K-Fold
        folds = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=np.random.randint(100000))
        list_folds = list(folds.split(X_train_used.values, y_train.values))

        preds_oof_temp = np.zeros((len(X_train), len(settings)))
        preds_test_temp = np.zeros((len(X_test), len(settings)))
        preds_train_temp = np.zeros((len(X_train), len(settings)))
        scores = []

        for j, setting in enumerate(settings):
            print(f'\nSetting: {setting}')
            
            # Optional: use setting to modify params if needed, e.g.,
            # params['num_leaves'] = setting

            for k, (trn_idx, val_idx) in enumerate(list_folds):
                trn_data = lgb.Dataset(X_train_used.iloc[trn_idx][features_train],
                                       label=y_train.iloc[trn_idx])
                val_data = lgb.Dataset(X_train_used.iloc[val_idx][features_train],
                                       label=y_train.iloc[val_idx])

                clf = lgb.train(
                    params,
                    trn_data,
                    num_boost_round=2000,
                    valid_sets=[trn_data, val_data],
                    callbacks=[
                        lgb.early_stopping(stopping_rounds=100, verbose=True),
                        lgb.log_evaluation(period=100)  # Print every 100 iterations
                    ]
                )

                # Predictions
                pred_val = clf.predict(X_train_used.iloc[val_idx][features_train])
                pred_test = clf.predict(X_test_used[features_train])
                pred_train = clf.predict(X_train_used.iloc[trn_idx][features_train])

                # Print metrics
                auc_val = roc_auc_score(y_train.iloc[val_idx], pred_val)
                loss_val = log_loss(y_train.iloc[val_idx], pred_val)
                auc_train = roc_auc_score(y_train.iloc[trn_idx], pred_train)
                loss_train = log_loss(y_train.iloc[trn_idx], pred_train)

                print(f"Fold {k+1} - val AUC: {auc_val:.4f} - loss: {loss_val*1000:.3f} "
                      f"- train AUC: {auc_train:.4f} - loss: {loss_train*1000:.3f}")

                # Store predictions
                preds_oof_temp[val_idx, j] += pred_val
                preds_test_temp[:, j] += pred_test / n_folds
                preds_train_temp[trn_idx, j] += pred_train / (n_folds-1)

            # Compute score per setting
            score_setting = roc_auc_score(y_train, preds_oof_temp[:, j])
            score_setting_log = log_loss(y_train, preds_oof_temp[:, j])
            scores.append(score_setting_log)
            print(f"Score setting: val AUC {score_setting:.4f}, loss {score_setting_log:.4f}")

        # Select best setting
        best_ind = np.argmin(scores)
        settings_best_ind.append(best_ind)
        preds_oof[:, i] = preds_oof_temp[:, best_ind]
        preds_test[:, i] = preds_test_temp[:, best_ind]
        preds_train[:, i] = preds_train_temp[:, best_ind]

        # Cumulative metrics
        preds_oof_cum = preds_oof[:, :i+1].mean(axis=1)
        print(f"Cumulative OOF AUC: {roc_auc_score(y_train, preds_oof_cum):.4f} - "
              f"Loss: {log_loss(y_train, preds_oof_cum):.4f}")
    
    return preds_oof, preds_test, preds_train


In [9]:
preds_oof, preds_test, preds_train = train_trees(X_train, y_train, X_test, features_used, params)

Training on features: ['var_0', 'var_0_count']

Setting: 4
[LightGBM] [Info] Number of positive: 12863, number of negative: 115137
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000743 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 268
[LightGBM] [Info] Number of data points in the train set: 128000, number of used features: 2
Training until validation scores don't improve for 100 rounds
[100]	training's binary_logloss: 0.323767	valid_1's binary_logloss: 0.323263
[200]	training's binary_logloss: 0.323488	valid_1's binary_logloss: 0.323273
Early stopping, best iteration is:
[165]	training's binary_logloss: 0.32357	valid_1's binary_logloss: 0.32325
Fold 1 - val AUC: 0.5567 - loss: 323.250 - train AUC: 0.5617 - loss: 323.570
[LightGBM] [Info] Number of positive: 12863, number of negative: 115137
[LightGBM] [Info] Auto-choosing row-wise m

In [12]:
# Assuming your targets are y_train and y_test
preds_oof_cum = np.zeros(preds_oof.shape[0])
preds_train_cum = np.zeros(preds_train.shape[0])
use_experimental = 'preds_test' in locals()  # or define manually
if use_experimental:
    preds_test_cum = np.zeros(preds_test.shape[0])

for i in range(preds_oof.shape[1]):
    preds_oof_cum += preds_oof[:, i]
    preds_train_cum += preds_train[:, i]
    
    # Average over features so far
    oof_mean = preds_oof_cum / (i + 1)
    train_mean = preds_train_cum / (i + 1)
    
    print(f"var_{i} Cum val AUC: {roc_auc_score(y_train, oof_mean):<8.5f}", end="")
    
    if use_experimental:
        preds_test_cum += preds_test[:, i]
        test_mean = preds_test_cum / (i + 1)
        print(f" - test AUC: {roc_auc_score(y_test, test_mean):<8.5f}", end="")
    
    print(f" - train AUC: {roc_auc_score(y_train, train_mean):<8.5f}")


var_0 Cum val AUC: 0.54805  - test AUC: 0.54463  - train AUC: 0.56151 
var_1 Cum val AUC: 0.57333  - test AUC: 0.56986  - train AUC: 0.58362 
var_2 Cum val AUC: 0.59575  - test AUC: 0.59218  - train AUC: 0.60673 
var_3 Cum val AUC: 0.59656  - test AUC: 0.59296  - train AUC: 0.61033 
var_4 Cum val AUC: 0.59720  - test AUC: 0.59343  - train AUC: 0.61302 
var_5 Cum val AUC: 0.60566  - test AUC: 0.59907  - train AUC: 0.62339 
var_6 Cum val AUC: 0.62596  - test AUC: 0.61724  - train AUC: 0.64149 
var_7 Cum val AUC: 0.62612  - test AUC: 0.61712  - train AUC: 0.64211 
var_8 Cum val AUC: 0.62781  - test AUC: 0.61929  - train AUC: 0.64560 
var_9 Cum val AUC: 0.63781  - test AUC: 0.63023  - train AUC: 0.65600 
var_10 Cum val AUC: 0.63794  - test AUC: 0.63038  - train AUC: 0.65626 
var_11 Cum val AUC: 0.64006  - test AUC: 0.63256  - train AUC: 0.65945 
var_12 Cum val AUC: 0.65638  - test AUC: 0.65282  - train AUC: 0.67472 
var_13 Cum val AUC: 0.66712  - test AUC: 0.66586  - train AUC: 0.68530 
va

In [13]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tensorflow import keras

n_splits = 7
num_preds = 5
epochs = 60
learning_rate_init = 0.02
batch_size = 4000

# Use the same y_train and y_test as your LightGBM code
# Use OOF/test/train preds from LightGBM
# preds_oof, preds_test, preds_train are outputs from train_trees

num_features = preds_oof.shape[1]  # number of features from LGBM

def get_features(preds):
    # Simply stack preds to shape (num_samples, num_features*num_preds)
    # If num_preds > 1, you can repeat or shift columns as needed
    return np.tile(preds, (1, num_preds))

def get_model_3():
    inp = keras.layers.Input((num_features*num_preds,))
    x = keras.layers.Reshape((num_features*num_preds,1))(inp)
    x = keras.layers.Conv1D(32, num_preds, strides=num_preds, activation='elu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Conv1D(24, 1, activation='elu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Conv1D(16, 1, activation='elu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Conv1D(4, 1, activation='elu')(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Reshape((num_features*4,1))(x)
    x = keras.layers.AveragePooling1D(2)(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.BatchNormalization()(x)
    out = keras.layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inputs=inp, outputs=out)

def lr_scheduler(epoch):
    if epoch <= epochs*0.8:
        return learning_rate_init
    else:
        return learning_rate_init * 0.1

def train_NN(features_oof, features_test, y_train, y_test=None):
    folds = StratifiedKFold(n_splits=n_splits)
    
    preds_nn_oof = np.zeros(features_oof.shape[0])
    preds_nn_test = np.zeros(features_test.shape[0])
    
    for trn_idx, val_idx in folds.split(features_oof, y_train):
        X_tr, X_val = features_oof[trn_idx], features_oof[val_idx]
        y_tr, y_val = y_train.values[trn_idx], y_train.values[val_idx]
        
        model = get_model_3()
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate_init, decay=1e-5)
        model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            verbose=2,
            callbacks=[keras.callbacks.LearningRateScheduler(lr_scheduler)]
        )
        
        preds_nn_oof[val_idx] = model.predict(X_val, batch_size=2000)[:,0]
        preds_nn_test += model.predict(features_test, batch_size=2000)[:,0] / n_splits
        
        print(f"Fold AUC: {roc_auc_score(y_train, preds_nn_oof):.5f}")
        if y_test is not None:
            print(f"Test AUC: {roc_auc_score(y_test, preds_nn_test):.5f}")
    
    return preds_nn_oof, preds_nn_test

# Prepare features for NN
features_oof = get_features(preds_oof)
features_test = get_features(preds_test)

# Train NN using LightGBM predictions as features
preds_nn_oof, preds_nn_test = train_NN(features_oof, features_test, y_train, y_test if use_experimental else None)

print("Final OOF AUC:", roc_auc_score(y_train, preds_nn_oof))
if use_experimental:
    print("Final Test AUC:", roc_auc_score(y_test, preds_nn_test))


2026-02-07 21:04:06.235701: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-07 21:04:06.355197: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770494646.413089 2766498 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770494646.429737 2766498 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770494646.528896 2766498 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Epoch 1/60


I0000 00:00:1770494663.233939 2820265 service.cc:152] XLA service 0x7f3434016c20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770494663.234136 2820265 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-02-07 21:04:23.310642: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1770494663.950801 2820265 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-02-07 21:04:24.638935: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 63.38GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-02-07 21:04:24.821225: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_

35/35 - 19s - 531ms/step - accuracy: 0.8984 - loss: 0.2554 - val_accuracy: 0.1005 - val_loss: 43.8818 - learning_rate: 0.0200
Epoch 2/60
35/35 - 2s - 44ms/step - accuracy: 0.9282 - loss: 0.1940 - val_accuracy: 0.1005 - val_loss: 33.7115 - learning_rate: 0.0200
Epoch 3/60
35/35 - 1s - 41ms/step - accuracy: 0.9291 - loss: 0.1921 - val_accuracy: 0.1005 - val_loss: 27.9098 - learning_rate: 0.0200
Epoch 4/60
35/35 - 1s - 41ms/step - accuracy: 0.9291 - loss: 0.1914 - val_accuracy: 0.1005 - val_loss: 28.5032 - learning_rate: 0.0200
Epoch 5/60
35/35 - 1s - 40ms/step - accuracy: 0.9297 - loss: 0.1916 - val_accuracy: 0.1005 - val_loss: 23.9378 - learning_rate: 0.0200
Epoch 6/60
35/35 - 1s - 39ms/step - accuracy: 0.9291 - loss: 0.1915 - val_accuracy: 0.1005 - val_loss: 25.2684 - learning_rate: 0.0200
Epoch 7/60
35/35 - 1s - 38ms/step - accuracy: 0.9298 - loss: 0.1907 - val_accuracy: 0.1005 - val_loss: 23.5683 - learning_rate: 0.0200
Epoch 8/60
35/35 - 1s - 38ms/step - accuracy: 0.9292 - loss: 0.1